# AgentCore Registry Demo — Tool Catalog for Open Insurance

This notebook demonstrates the **AWS AgentCore Registry** — a managed discovery and catalog service for MCP servers. While the Gateway **runs** your tools, the Registry **catalogs** them so agents can discover approved tools across your organization.

## Why does the Registry matter?

Without a Registry, agents have **hardcoded tool lists** — every team manually configures which MCP servers their agents can see. This breaks down at scale:

- **Team A** builds a PolicyLookup tool but **Team B** doesn't know it exists, so they build their own
- A new tool is deployed but **agents can't find it** without someone updating their config
- There's **no governance** — anyone can publish anything, with no review or approval process
- When a tool is deprecated, agents still try to call it because **nobody updated the hardcoded list**

The Registry solves this with a **Publish → Discover → Govern** workflow:

```
Teams publish MCP server records     Agents search with natural language
        │                                       │
        ▼                                       ▼
   ┌──────────┐                          "Find tools for
   │ Registry │  ◀── semantic search ──  insurance claims"
   │          │                                 │
   │ PolicyLookup (v1.0, APPROVED)              │
   │ ClaimsData  (v1.0, APPROVED)               ▼
   │ CRMTools    (v2.1, DEPRECATED)     Returns matching records
   │ NewTool     (v0.1, DRAFT)          with tool schemas
   └──────────┘
        │
        │  Governance: DRAFT → review → APPROVED
        │  Deprecated tools hidden from search
        │  EventBridge integration for approval workflows
```

**Key point:** The Registry is a **catalog**, not a runtime. It stores tool metadata (names, descriptions, input schemas) — it does NOT invoke tools. Agents discover tools via the Registry, then invoke them through the **Gateway** (which handles authentication, Cedar policies, and Lambda execution).

| Layer | Role | Analogy |
|-------|------|---------|
| **Registry** | Find the right tool | App store / internal wiki |
| **Gateway** | Run the tool securely | API gateway / runtime |

## Architecture

```
Claude Code / Strands Agent
  │
  ├── MCP ──▶ Registry (discover tools — metadata only)
  │           │  search_registry_records tool
  │           │  JWT auth (Okta)
  │           │
  │           └── Returns: tool names, schemas, descriptions
  │
  ├── MCP ──▶ Gateway (invoke tools — actual execution)
  │           │  PolicyLookup___lookup_policy
  │           │  ClaimsData___query_claims
  │           │  JWT auth (Okta) + Cedar ENFORCE
  │           │
  │           ├───────────────┐
  │           ▼               ▼
  │         Lambda          Lambda
  │       (PolicyLookup)  (ClaimsData)
  │
  ▼
Okta (OIDC identity provider)
```

## What this notebook does

1. Install dependencies and load existing Gateway config
2. Create a **Registry** with Okta JWT authorization (same IdP as Gateway)
3. Publish **MCP server records** for PolicyLookup and ClaimsData tools
4. Demo **governance workflow** — submit for approval, approve records
5. **Search** the registry using Okta JWT tokens (agent path)
6. Configure **Claude Code** to connect to the Registry MCP endpoint
7. Save config and provide test instructions

## Prerequisites

- **`01_setup.ipynb`** completed (Gateway, Lambda targets, Cedar policies deployed)
- **`gateway_config.json`** exists with deployed resource IDs
- **`.env`** with Okta credentials (OKTA_API_TOKEN, user passwords)
- **boto3 >= 1.42.89** (upgraded in Cell 1)

## Cell 1: Dependencies + Configuration

Upgrades boto3 for Registry API support and loads the existing Gateway config from `01_setup.ipynb`.

In [2]:
%pip install -q --upgrade boto3 requests python-dotenv PyJWT

import os
import json
import time
import boto3
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

# --- Load Gateway config from 01_setup ---
with open("gateway_config.json") as f:
    config = json.load(f)

GATEWAY_ID = config["gateway_id"]
GATEWAY_URL = config["gateway_url"]
OKTA_DOMAIN = config["okta_domain"]
OKTA_ISSUER = config["okta_issuer"]
OKTA_CLIENT_ID = config["okta_client_id"]       # Web app (confidential client)
AWS_REGION = config["aws_region"]

# --- Okta credentials from .env ---
OKTA_CLIENT_SECRET = os.environ["OKTA_CLIENT_SECRET"]
OKTA_API_TOKEN = os.environ.get("OKTA_API_TOKEN", "")
BOB_USERNAME = os.environ.get("BOB_USERNAME", "")
BOB_PASSWORD = os.environ.get("BOB_PASSWORD", "")

# --- SPA client from notebook 03 (optional) ---
SPA_CLIENT_ID = config.get("okta_spa_client_id", "")
CALLBACK_PORT = config.get("claude_code_callback_port", 8400)

# --- boto3 clients ---
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)
agentcore_data = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

# --- Verify Registry API is available ---
assert hasattr(agentcore_control, "create_registry"), \
    f"boto3 {boto3.__version__} lacks Registry API — need >= 1.42.89"

print(f"boto3 version:   {boto3.__version__}")
print(f"Gateway ID:      {GATEWAY_ID}")
print(f"Okta Domain:     {OKTA_DOMAIN}")
print(f"Okta Issuer:     {OKTA_ISSUER}")
print(f"Web Client ID:   {OKTA_CLIENT_ID}")
print(f"AWS Region:      {AWS_REGION}")
if SPA_CLIENT_ID:
    print(f"SPA Client ID:   {SPA_CLIENT_ID} (from notebook 03)")
else:
    print(f"SPA Client ID:   (not set — run 03_claude_code_oauth_demo.ipynb for Claude Code integration)")
print(f"\n✅ Configuration loaded")

Note: you may need to restart the kernel to use updated packages.
boto3 version:   1.42.89
Gateway ID:      agentcore-mcp-demo-gateway-r68b4sket5
Okta Domain:     integrator-5723923.okta.com
Okta Issuer:     https://integrator-5723923.okta.com/oauth2/default
Web Client ID:   0oa119ott9ebnV75Z698
AWS Region:      ap-southeast-2
SPA Client ID:   0oa11b5p776R7xW9t698 (from notebook 03)

✅ Configuration loaded


## Cell 2: Create the Registry

Creates an AgentCore Registry with **Okta JWT authorization** — the same identity provider used by the Gateway. This means agents authenticate once with Okta and can both discover tools (Registry) and invoke them (Gateway).

Key settings:
- **`authorizerType: CUSTOM_JWT`** — validates Okta JWTs against the OIDC discovery endpoint
- **`allowedAudience: ["api://default"]`** — Okta's default authorization server audience
- **`allowedClients`** — accepts tokens from both the web app and SPA clients
- **`autoApproval: False`** — requires explicit approval before records are searchable (governance demo)

In [5]:
REGISTRY_NAME = "open-insurance-registry"
OKTA_DISCOVERY_URL = f"{OKTA_ISSUER}/.well-known/openid-configuration"

# Build allowedClients list — web app always, SPA if available
allowed_clients = [OKTA_CLIENT_ID]
if SPA_CLIENT_ID and SPA_CLIENT_ID not in allowed_clients:
    allowed_clients.append(SPA_CLIENT_ID)

print(f"Registry name:    {REGISTRY_NAME}")
print(f"Discovery URL:    {OKTA_DISCOVERY_URL}")
print(f"Allowed clients:  {allowed_clients}")

# --- Create the Registry ---
print(f"\nCreating Registry with Okta JWT auth...")

try:
    resp = agentcore_control.create_registry(
        name=REGISTRY_NAME,
        description="Open Insurance tool catalog — discover MCP servers for policy and claims operations",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": OKTA_DISCOVERY_URL,
                "allowedAudience": ["api://default"],
                "allowedClients": allowed_clients,
            }
        },
        approvalConfiguration={"autoApproval": False},
    )
    REGISTRY_ARN = resp["registryArn"]
    # Extract registry ID from ARN (format: arn:aws:bedrock-agentcore:region:account:registry/registryId)
    REGISTRY_ID = REGISTRY_ARN.rsplit("/", 1)[-1]
    print(f"Created Registry: {REGISTRY_ID}")
except agentcore_control.exceptions.ConflictException:
    # Registry already exists — find it
    print("Registry already exists — looking up...")
    registries = agentcore_control.list_registries()
    for reg in registries.get("registries", registries.get("items", [])):
        if reg["name"] == REGISTRY_NAME:
            REGISTRY_ID = reg["registryId"]
            REGISTRY_ARN = reg["registryArn"]
            print(f"Found existing Registry: {REGISTRY_ID}")
            break
    else:
        raise RuntimeError(f"ConflictException but could not find registry '{REGISTRY_NAME}'")

# --- Poll for READY status ---
print("Waiting for Registry to be READY...")
for attempt in range(30):
    reg = agentcore_control.get_registry(registryId=REGISTRY_ID)
    status = reg["status"]
    if status == "READY":
        REGISTRY_ARN = reg["registryArn"]
        print(f"\n✅ Registry READY")
        print(f"  Registry ID:  {REGISTRY_ID}")
        print(f"  Registry ARN: {REGISTRY_ARN}")
        break
    elif status in ("FAILED", "CREATE_FAILED"):
        print(f"Registry FAILED: {reg}")
        break
    time.sleep(5)
    print(f"  Status: {status} ({(attempt + 1) * 5}s)")
else:
    print("Warning: Registry did not become READY in time")

Registry name:    open-insurance-registry
Discovery URL:    https://integrator-5723923.okta.com/oauth2/default/.well-known/openid-configuration
Allowed clients:  ['0oa119ott9ebnV75Z698', '0oa11b5p776R7xW9t698']

Creating Registry with Okta JWT auth...
Created Registry: 2waqSwYyQPIhl7pD
Waiting for Registry to be READY...
  Status: CREATING (5s)
  Status: CREATING (10s)
  Status: CREATING (15s)
  Status: CREATING (20s)
  Status: CREATING (25s)
  Status: CREATING (30s)
  Status: CREATING (35s)
  Status: CREATING (40s)
  Status: CREATING (45s)
  Status: CREATING (50s)
  Status: CREATING (55s)
  Status: CREATING (60s)
  Status: CREATING (65s)
  Status: CREATING (70s)
  Status: CREATING (75s)
  Status: CREATING (80s)
  Status: CREATING (85s)
  Status: CREATING (90s)

✅ Registry READY
  Registry ID:  2waqSwYyQPIhl7pD
  Registry ARN: arn:aws:bedrock-agentcore:ap-southeast-2:126672810070:registry/2waqSwYyQPIhl7pD


## Cell 3: Publish MCP Server Records

Each Registry record represents an MCP server with its tool schemas. We register the same tools that are deployed as Gateway targets:

| Record | Tool | Description | Gateway Target |
|--------|------|-------------|----------------|
| `open-insurance-policy-lookup` | `lookup_policy` | Policy details by number | PolicyLookup Lambda |
| `open-insurance-claims-data` | `query_claims` | Confidential claims query | ClaimsData Lambda |

Records start in **DRAFT** status — they are not searchable until approved (Cell 4).

### MCP Descriptor Format

Each record carries two descriptors:
- **`server`** — MCP server metadata (name, description, version)
- **`tools`** — Tool definitions with input schemas (same format as Gateway target `inlinePayload`)

In [15]:
# --- Tool schemas (same as Gateway targets in 01_setup) ---

POLICY_LOOKUP_TOOL = {
    "name": "lookup_policy",
    "description": "Look up insurance policy details by policy number. Available policies: POL-10042, POL-10078, POL-10103, POL-10156, POL-10201.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "policy_number": {"type": "string", "description": "Policy number (e.g., POL-10042)"}
        },
        "required": ["policy_number"]
    }
}

CLAIMS_TOOL = {
    "name": "query_claims",
    "description": "Query insurance claims data. Contains CONFIDENTIAL claims information including adjuster notes. Search by claim ID, policy number, holder name, or ask for 'claims summary'. Use max_amount to filter by claim size.",
    "inputSchema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Claims query — e.g., 'claims summary', 'CLM-2024-0891', 'POL-10042', or a holder name"},
            "max_amount": {"type": "integer", "description": "Maximum claim amount to return (e.g., 100000 for claims <= $100K)"}
        },
        "required": ["query"]
    }
}

# --- Define registry records ---
# Record-level description (free text) includes Gateway connection info.
# Descriptor inlineContent must conform to MCP schema — kept minimal.

records_to_create = [
    {
        "name": "open-insurance-policy-lookup",
        "description": (
            "Insurance policy lookup MCP server — returns policy details by policy number. "
            "Available to all authenticated users. "
            f"Hosted on AgentCore Gateway: {GATEWAY_URL} "
            "(MCP StreamableHTTP, Okta JWT auth, Cedar ENFORCE mode). "
            "Tool name on Gateway: PolicyLookup___lookup_policy."
        ),
        "server_name": "open-insurance/policy-lookup",
        "server_description": "Insurance policy lookup MCP server for the Open Insurance platform",
        "tools": [POLICY_LOOKUP_TOOL],
    },
    {
        "name": "open-insurance-claims-data",
        "description": (
            "Confidential insurance claims data MCP server — restricted to authorized claims adjusters "
            "via Cedar policies. Includes adjuster notes, fraud flags, and claim amounts up to $250K. "
            f"Hosted on AgentCore Gateway: {GATEWAY_URL} "
            "(MCP StreamableHTTP, Okta JWT auth, Cedar ENFORCE mode). "
            "Tool name on Gateway: ClaimsData___query_claims. "
            "Access restricted by Cedar policy — only claims adjusters (e.g., Bob) have full access."
        ),
        "server_name": "open-insurance/claims-data",
        "server_description": "Insurance claims data MCP server with confidential adjuster notes",
        "tools": [CLAIMS_TOOL],
    },
]

# --- Create records ---
record_ids = {}

for rec in records_to_create:
    try:
        resp = agentcore_control.create_registry_record(
            registryId=REGISTRY_ID,
            name=rec["name"],
            description=rec["description"],
            descriptorType="MCP",
            recordVersion="1.0.0",
            descriptors={
                "mcp": {
                    "server": {
                        "schemaVersion": "2025-12-11",
                        "inlineContent": json.dumps({
                            "name": rec["server_name"],
                            "description": rec["server_description"],
                            "version": "1.0.0"
                        })
                    },
                    "tools": {
                        "protocolVersion": "2025-11-25",
                        "inlineContent": json.dumps({"tools": rec["tools"]})
                    }
                }
            },
        )
        # Response only has recordArn — extract ID from it
        record_arn = resp["recordArn"]
        record_id = record_arn.rsplit("/", 1)[-1]
        record_ids[rec["name"]] = record_id
        print(f"Created record: {rec['name']} [{record_id}] — status: {resp.get('status', 'DRAFT')}")
    except agentcore_control.exceptions.ConflictException:
        # Record exists — find it
        existing = agentcore_control.list_registry_records(registryId=REGISTRY_ID)
        for r in existing.get("registryRecords", existing.get("items", [])):
            if r["name"] == rec["name"]:
                record_ids[rec["name"]] = r["recordId"]
                print(f"Record already exists: {rec['name']} [{r['recordId']}] — status: {r['status']}")
                break

print(f"\n✅ {len(record_ids)} records created/found")
for name, rid in record_ids.items():
    print(f"  {name}: {rid}")

Created record: open-insurance-policy-lookup [GCuPEykyxkH9] — status: CREATING
Created record: open-insurance-claims-data [pg3g4OVR2HgH] — status: CREATING

✅ 2 records created/found
  open-insurance-policy-lookup: GCuPEykyxkH9
  open-insurance-claims-data: pg3g4OVR2HgH


## Cell 4: Governance — Approval Workflow

Records must be **approved** before they appear in search results. This simulates an enterprise governance workflow:

```
Developer publishes record → DRAFT
    ↓
Developer submits for review → PENDING_APPROVAL
    ↓
Team lead / curator approves → APPROVED (now searchable)
```

In production, the submit step triggers an **EventBridge** event that can integrate with ticketing systems or security review workflows. Here we play both roles (publisher + curator) to demo the lifecycle.

In [16]:
for name, record_id in record_ids.items():
    # Check current status
    record = agentcore_control.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    status = record["status"]
    print(f"\n--- {name} ---")
    print(f"  Current status: {status}")

    # Step 1: Submit for approval (DRAFT → PENDING_APPROVAL)
    if status == "DRAFT":
        agentcore_control.submit_registry_record_for_approval(
            registryId=REGISTRY_ID, recordId=record_id
        )
        print(f"  Submitted for approval: DRAFT → PENDING_APPROVAL")
        status = "PENDING_APPROVAL"

    # Step 2: Approve (PENDING_APPROVAL → APPROVED)
    if status == "PENDING_APPROVAL":
        agentcore_control.update_registry_record_status(
            registryId=REGISTRY_ID,
            recordId=record_id,
            status="APPROVED",
            statusReason="Approved for Open Insurance tool catalog demo",
        )
        print(f"  Approved: PENDING_APPROVAL → APPROVED")
        status = "APPROVED"

    if status == "APPROVED":
        print(f"  ✅ Record is APPROVED and searchable")
    else:
        print(f"  ⚠️  Unexpected status: {status}")

# Verify final state
print(f"\n--- Final Record Status ---")
records = agentcore_control.list_registry_records(registryId=REGISTRY_ID)
for r in records.get("registryRecords", records.get("items", [])):
    icon = "✅" if r["status"] == "APPROVED" else "⚠️"
    print(f"  {icon} {r['name']}: {r['status']}")

print(f"\n✅ All records approved — they will appear in search results shortly")


--- open-insurance-policy-lookup ---
  Current status: DRAFT
  Submitted for approval: DRAFT → PENDING_APPROVAL
  Approved: PENDING_APPROVAL → APPROVED
  ✅ Record is APPROVED and searchable

--- open-insurance-claims-data ---
  Current status: DRAFT
  Submitted for approval: DRAFT → PENDING_APPROVAL
  Approved: PENDING_APPROVAL → APPROVED
  ✅ Record is APPROVED and searchable

--- Final Record Status ---
  ⚠️ open-insurance-claims-data: DEPRECATED
  ⚠️ open-insurance-policy-lookup: DEPRECATED
  ✅ open-insurance-claims-data: APPROVED
  ✅ open-insurance-policy-lookup: APPROVED
  ⚠️ open-insurance-policy-lookup: DEPRECATED

✅ All records approved — they will appear in search results shortly


## Cell 5: Search the Registry (JWT Auth — Okta Token)

Since the Registry uses **CUSTOM_JWT** authorization, the search API requires a JWT token — IAM credentials won't work for data plane operations. This matches the agent experience: authenticate with your corporate identity, then search for tools.

We get a JWT via Okta's ROPC flow (same as `02_demo.ipynb`), then use it for both:
- **boto3 search** (with a custom event handler to inject the Bearer token)
- **Raw HTTP search** (direct `requests` call)

The Registry supports **hybrid search** — semantic (conceptual intent) and keyword matching run in parallel, with results ranked by combined relevance.

> **Note:** Newly approved records have an eventual consistency delay (seconds to minutes). We retry with backoff if results are empty.

In [17]:
import jwt as pyjwt  # PyJWT — for decoding (not verifying) token contents

def display_results(results, query):
    """Display search results with tool schemas."""
    print(f"\n{'='*60}")
    print(f"Search: \"{query}\"  →  {len(results)} result(s)")
    print(f"{'='*60}")
    for i, r in enumerate(results, 1):
        print(f"\n  [{i}] {r['name']}")
        print(f"      Type: {r['descriptorType']}  |  Status: {r['status']}  |  Version: {r.get('version', 'N/A')}")
        print(f"      Description: {r.get('description', 'N/A')[:100]}")
        # Parse and show embedded tool schemas
        if r.get("descriptors", {}).get("mcp", {}).get("tools"):
            tools_content = r["descriptors"]["mcp"]["tools"].get("inlineContent", "{}")
            tools = json.loads(tools_content).get("tools", [])
            for t in tools:
                print(f"      Tool: {t['name']} — {t['description'][:80]}...")
                props = t.get("inputSchema", {}).get("properties", {})
                if props:
                    print(f"            Inputs: {', '.join(props.keys())}")


# --- Step 1: Get Okta JWT via ROPC flow ---
if not BOB_USERNAME or not BOB_PASSWORD:
    raise SystemExit("BOB_USERNAME / BOB_PASSWORD not set in .env — required for JWT search")

print("Authenticating Bob via Okta ROPC flow...")
token_resp = requests.post(
    f"{OKTA_ISSUER}/v1/token",
    data={
        "grant_type": "password",
        "username": BOB_USERNAME,
        "password": BOB_PASSWORD,
        "client_id": OKTA_CLIENT_ID,
        "client_secret": OKTA_CLIENT_SECRET,
        "scope": "openid profile groups",
    },
)

if token_resp.status_code != 200:
    raise RuntimeError(f"Token error: {token_resp.status_code} — {token_resp.text[:200]}")

access_token = token_resp.json()["access_token"]
claims = pyjwt.decode(access_token, options={"verify_signature": False})
print(f"  Authenticated as: {claims.get('sub', 'unknown')}")
print(f"  Client ID:        {claims.get('cid', claims.get('client_id', 'N/A'))}")
print(f"  Audience:         {claims.get('aud', 'N/A')}")

# --- Step 2: Search registry with Bearer token via HTTP ---
REGISTRY_SEARCH_URL = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/registry-records/search"

def search_registry_jwt(query, max_results=10):
    """Search registry with JWT Bearer token, with retry for eventual consistency."""
    for attempt in range(5):
        resp = requests.post(
            REGISTRY_SEARCH_URL,
            headers={
                "Authorization": f"Bearer {access_token}",
                "Content-Type": "application/json",
            },
            json={
                "searchQuery": query,
                "registryIds": [REGISTRY_ARN],
                "maxResults": max_results,
            },
        )
        if resp.status_code != 200:
            print(f"  Search error: {resp.status_code} — {resp.text[:200]}")
            return []
        results = resp.json().get("registryRecords", [])
        if results:
            return results
        wait = 5 * (attempt + 1)
        print(f"  No results yet (eventual consistency) — retrying in {wait}s...")
        time.sleep(wait)
    return []


print(f"\nSearching registry via JWT Bearer token...")
print(f"  Endpoint: {REGISTRY_SEARCH_URL}")

# --- Search 1: Broad semantic search ---
results_all = search_registry_jwt("insurance")
display_results(results_all, "insurance")

# --- Search 2: Targeted keyword search ---
results_policy = search_registry_jwt("policy lookup")
display_results(results_policy, "policy lookup")

# --- Search 3: Semantic search for restricted data ---
results_claims = search_registry_jwt("confidential claims adjuster")
display_results(results_claims, "confidential claims adjuster")

print(f"\n✅ JWT search working — {len(results_all)} total records found")
print(f"   Agents can discover tools with just an Okta token — no IAM credentials needed")

Authenticating Bob via Okta ROPC flow...
  Authenticated as: bob@integrator-5723923.okta.com
  Client ID:        0oa119ott9ebnV75Z698
  Audience:         api://default

Searching registry via JWT Bearer token...
  Endpoint: https://bedrock-agentcore.ap-southeast-2.amazonaws.com/registry-records/search

Search: "insurance"  →  2 result(s)

  [1] open-insurance-policy-lookup
      Type: MCP  |  Status: APPROVED  |  Version: 1.0.0
      Description: Insurance policy lookup MCP server — returns policy details by policy number. Available to all authe
      Tool: lookup_policy — Look up insurance policy details by policy number. Available policies: POL-10042...
            Inputs: policy_number

  [2] open-insurance-claims-data
      Type: MCP  |  Status: APPROVED  |  Version: 1.0.0
      Description: Confidential insurance claims data MCP server — restricted to authorized claims adjusters via Cedar 
      Tool: query_claims — Query insurance claims data. Contains CONFIDENTIAL claims informa

## Cell 6: Configure Claude Code for Registry-First Discovery

This configures Claude Code with **only the Registry** as an MCP server — the Gateway is intentionally left out. When Claude Code searches the Registry and finds tools hosted on the Gateway, it can dynamically add the Gateway connection and authenticate on demand.

This "Registry-first" pattern is valuable for **regulated industries**:
- Users only authenticate to backends they actually need (**least-privilege**)
- Different Gateways can require different Okta scopes or authorization servers
- Audit trails show explicit per-system consent, not blanket access
- The Registry becomes the **single front door** that guides agents to the right tools

### Demo Flow

```
1. Start Claude Code → authenticates to Registry (Okta popup)
2. Ask: "What insurance tools are available?"
   → Registry search returns tools + Gateway URL
3. Claude Code adds Gateway as MCP server
4. Run /mcp → authenticates to Gateway (Okta popup)
5. Ask: "Look up policy POL-10042"
   → Invokes tool on Gateway with Cedar enforcement
```

> **Note:** If `03_claude_code_oauth_demo.ipynb` already added the Gateway, this cell removes it to demonstrate the Registry-first flow. The Gateway config is saved and can be re-added later.

In [21]:
import subprocess

REGISTRY_MCP_URL = f"https://bedrock-agentcore.{AWS_REGION}.amazonaws.com/registry/{REGISTRY_ID}/mcp"

print(f"Registry MCP endpoint: {REGISTRY_MCP_URL}")

if not SPA_CLIENT_ID:
    print(f"\n⚠️  No SPA client ID found in gateway_config.json")
    print(f"   Run 03_claude_code_oauth_demo.ipynb first to create the Okta SPA app,")
    print(f"   then re-run this notebook from Cell 1.")
else:
    # --- Step 1: Ensure SPA client is in Registry's allowedClients ---
    reg = agentcore_control.get_registry(registryId=REGISTRY_ID)
    current_clients = reg.get("authorizerConfiguration", {}).get("customJWTAuthorizer", {}).get("allowedClients", [])

    if SPA_CLIENT_ID not in current_clients:
        print(f"\nAdding SPA client to Registry's allowedClients...")
        new_clients = current_clients + [SPA_CLIENT_ID]
        agentcore_control.update_registry(
            registryId=REGISTRY_ID,
            authorizerConfiguration={
                "customJWTAuthorizer": {
                    "discoveryUrl": OKTA_DISCOVERY_URL,
                    "allowedAudience": ["api://default"],
                    "allowedClients": new_clients,
                }
            },
        )
        print(f"  Updated allowedClients: {new_clients}")

        for attempt in range(12):
            reg = agentcore_control.get_registry(registryId=REGISTRY_ID)
            if reg["status"] == "READY":
                print(f"  Registry READY")
                break
            time.sleep(5)
            print(f"  Status: {reg['status']} ({(attempt + 1) * 5}s)")
    else:
        print(f"SPA client already in allowedClients")

    # --- Step 2: Remove Gateway from Claude Code (Registry-first demo) ---
    print(f"\nRemoving agentcore-gateway from Claude Code (Registry-first flow)...")
    subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)

    # --- Step 3: Add Registry MCP server ---
    MCP_SERVER_NAME = "open-insurance-registry"

    registry_mcp_config = {
        "type": "http",
        "url": REGISTRY_MCP_URL,
        "oauth": {
            "clientId": SPA_CLIENT_ID,
            "callbackPort": CALLBACK_PORT,
            "scope": "openid groups",
            "authorizationUrl": f"{OKTA_ISSUER}/v1/authorize",
            "tokenUrl": f"{OKTA_ISSUER}/v1/token",
        },
    }

    subprocess.run(["claude", "mcp", "remove", MCP_SERVER_NAME], capture_output=True)
    result = subprocess.run(
        ["claude", "mcp", "add-json", MCP_SERVER_NAME, json.dumps(registry_mcp_config)],
        capture_output=True, text=True,
    )

    if result.returncode == 0:
        print(f"✅ Added '{MCP_SERVER_NAME}' MCP server to Claude Code")
    else:
        print(f"⚠️  claude mcp add-json failed: {result.stderr}")

    # --- Step 4: Patch .claude.json for scope/URLs (same workaround as notebook 03) ---
    claude_json_path = os.path.expanduser("~/.claude.json")
    try:
        with open(claude_json_path) as f:
            claude_config = json.load(f)

        project_key = os.getcwd()
        if "projects" in claude_config and project_key in claude_config["projects"]:
            servers = claude_config["projects"][project_key].get("mcpServers", {})
            if MCP_SERVER_NAME in servers:
                servers[MCP_SERVER_NAME]["oauth"]["scope"] = "openid groups"
                servers[MCP_SERVER_NAME]["oauth"]["authorizationUrl"] = f"{OKTA_ISSUER}/v1/authorize"
                servers[MCP_SERVER_NAME]["oauth"]["tokenUrl"] = f"{OKTA_ISSUER}/v1/token"
                with open(claude_json_path, "w") as f:
                    json.dump(claude_config, f, indent=2)
                print("Patched .claude.json with scope and OAuth URLs")
    except FileNotFoundError:
        print("Note: ~/.claude.json not found — Claude Code config may be stored elsewhere")

    # --- Step 5: Save Gateway config for re-adding later ---
    gateway_mcp_config = {
        "type": "http",
        "url": GATEWAY_URL,
        "oauth": {
            "clientId": SPA_CLIENT_ID,
            "callbackPort": CALLBACK_PORT,
            "scope": "openid groups",
            "authorizationUrl": f"{OKTA_ISSUER}/v1/authorize",
            "tokenUrl": f"{OKTA_ISSUER}/v1/token",
        },
    }

    print(f"\n--- Claude Code MCP Servers ---")
    print(f"  ✅ open-insurance-registry:  {REGISTRY_MCP_URL}")
    print(f"  ❌ agentcore-gateway:        REMOVED (Registry-first demo)")
    print(f"\n--- Gateway config (for re-adding during demo) ---")
    print(f"  claude mcp add-json agentcore-gateway '{json.dumps(gateway_mcp_config)}'")
    print(f"\n  After adding, run /mcp in Claude Code to connect and authenticate.")

Registry MCP endpoint: https://bedrock-agentcore.ap-southeast-2.amazonaws.com/registry/2waqSwYyQPIhl7pD/mcp
SPA client already in allowedClients

Removing agentcore-gateway from Claude Code (Registry-first flow)...
✅ Added 'open-insurance-registry' MCP server to Claude Code
Patched .claude.json with scope and OAuth URLs

--- Claude Code MCP Servers ---
  ✅ open-insurance-registry:  https://bedrock-agentcore.ap-southeast-2.amazonaws.com/registry/2waqSwYyQPIhl7pD/mcp
  ❌ agentcore-gateway:        REMOVED (Registry-first demo)

--- Gateway config (for re-adding during demo) ---
  claude mcp add-json agentcore-gateway '{"type": "http", "url": "https://agentcore-mcp-demo-gateway-r68b4sket5.gateway.bedrock-agentcore.ap-southeast-2.amazonaws.com/mcp", "oauth": {"clientId": "0oa11b5p776R7xW9t698", "callbackPort": 8400, "scope": "openid groups", "authorizationUrl": "https://integrator-5723923.okta.com/oauth2/default/v1/authorize", "tokenUrl": "https://integrator-5723923.okta.com/oauth2/default/

## Cell 7: Save Configuration

Saves Registry resource IDs to `gateway_config.json` for reference and cleanup.

In [19]:
# Reload config to avoid overwriting changes from other notebooks
with open("gateway_config.json") as f:
    config = json.load(f)

config["registry_id"] = REGISTRY_ID
config["registry_arn"] = REGISTRY_ARN
config["registry_mcp_url"] = REGISTRY_MCP_URL
config["registry_records"] = record_ids

with open("gateway_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Config saved to gateway_config.json")
print(f"\n  registry_id:      {REGISTRY_ID}")
print(f"  registry_arn:     {REGISTRY_ARN}")
print(f"  registry_mcp_url: {REGISTRY_MCP_URL}")
print(f"  registry_records: {list(record_ids.keys())}")

✅ Config saved to gateway_config.json

  registry_id:      2waqSwYyQPIhl7pD
  registry_arn:     arn:aws:bedrock-agentcore:ap-southeast-2:126672810070:registry/2waqSwYyQPIhl7pD
  registry_mcp_url: https://bedrock-agentcore.ap-southeast-2.amazonaws.com/registry/2waqSwYyQPIhl7pD/mcp
  registry_records: ['open-insurance-policy-lookup', 'open-insurance-claims-data']


## Cell 8: Restore Claude Code to Gateway-Only Mode

After running the Registry-first demo (Cell 6), use this cell to **restore** Claude Code to the original Gateway-only setup — removing the Registry MCP server and re-adding the Gateway.

This lets you quickly toggle between demo modes:
- **Cell 6** → Registry-first (Registry only, no Gateway)
- **Cell 8** → Gateway-only (Gateway only, no Registry)

After running this cell, restart Claude Code and run `/mcp` to authenticate to the Gateway.

In [20]:
import os
import json
import subprocess

# --- Load config ---
with open("gateway_config.json") as f:
    cfg = json.load(f)

spa_client_id = cfg.get("okta_spa_client_id", "")
if not spa_client_id:
    raise SystemExit("No SPA client ID found — run 03_claude_code_oauth_demo.ipynb first")

# --- Step 1: Remove Registry MCP server ---
result = subprocess.run(["claude", "mcp", "remove", "open-insurance-registry"], capture_output=True, text=True)
if result.returncode == 0:
    print("Removed open-insurance-registry from Claude Code")
else:
    print(f"open-insurance-registry not found (may already be removed)")

# --- Step 2: Re-add Gateway MCP server ---
gateway_mcp_config = {
    "type": "http",
    "url": cfg["gateway_url"],
    "oauth": {
        "clientId": spa_client_id,
        "callbackPort": cfg.get("claude_code_callback_port", 8400),
        "scope": "openid groups",
        "authorizationUrl": f"{cfg['okta_issuer']}/v1/authorize",
        "tokenUrl": f"{cfg['okta_issuer']}/v1/token",
    },
}

subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)
result = subprocess.run(
    ["claude", "mcp", "add-json", "agentcore-gateway", json.dumps(gateway_mcp_config)],
    capture_output=True, text=True,
)

if result.returncode == 0:
    print("Added agentcore-gateway to Claude Code")
else:
    print(f"⚠️  Failed to add gateway: {result.stderr}")

# --- Step 3: Patch .claude.json for scope/URLs ---
claude_json_path = os.path.expanduser("~/.claude.json")
try:
    with open(claude_json_path) as f:
        claude_config = json.load(f)

    project_key = os.getcwd()
    if "projects" in claude_config and project_key in claude_config["projects"]:
        servers = claude_config["projects"][project_key].get("mcpServers", {})
        if "agentcore-gateway" in servers:
            servers["agentcore-gateway"]["oauth"]["scope"] = "openid groups"
            servers["agentcore-gateway"]["oauth"]["authorizationUrl"] = f"{cfg['okta_issuer']}/v1/authorize"
            servers["agentcore-gateway"]["oauth"]["tokenUrl"] = f"{cfg['okta_issuer']}/v1/token"
            with open(claude_json_path, "w") as f:
                json.dump(claude_config, f, indent=2)
            print("Patched .claude.json with scope and OAuth URLs")
except FileNotFoundError:
    print("Note: ~/.claude.json not found")

# --- Summary ---
print(f"\n--- Claude Code MCP Servers ---")
print(f"  ✅ agentcore-gateway:        {cfg['gateway_url']}")
print(f"  ❌ open-insurance-registry:  REMOVED")
print(f"\nRestart Claude Code and run /mcp to connect and authenticate.")

Removed open-insurance-registry from Claude Code
Added agentcore-gateway to Claude Code
Patched .claude.json with scope and OAuth URLs

--- Claude Code MCP Servers ---
  ✅ agentcore-gateway:        https://agentcore-mcp-demo-gateway-r68b4sket5.gateway.bedrock-agentcore.ap-southeast-2.amazonaws.com/mcp
  ❌ open-insurance-registry:  REMOVED

Restart Claude Code and run /mcp to connect and authenticate.


## Test It: Registry-First Discovery Flow

Clear OAuth tokens and start a fresh Claude Code session:

```bash
rm -rf ~/.claude/oauth-tokens/
cd /path/to/demo-centralized-mcp-server
claude
```

### Step 1: Authenticate to Registry only

On startup, Claude Code connects to `open-insurance-registry` — **Okta browser popup** for authentication. The Gateway is **not connected** — only the Registry.

Run `/mcp` to verify:
- `open-insurance-registry` — connected
- `agentcore-gateway` — **not listed**

### Step 2: Discover tools via Registry

```
> What insurance tools are available? Search the registry.
```

Claude Code uses `search_registry_records` and gets back:
- **open-insurance/policy-lookup** — hosted at Gateway URL, tool name `PolicyLookup___lookup_policy`
- **open-insurance/claims-data** — hosted at Gateway URL, restricted to claims adjusters

At this point, Claude Code knows **what tools exist** and **where they're hosted**, but it **can't invoke them** — no Gateway connection yet.

### Step 3: Add the Gateway connection

Claude Code sees the Gateway URL in the Registry results. Ask it to add the connection:

```
> I need to use the policy lookup tool. Can you add the AgentCore Gateway as an MCP server?
```

Claude Code runs `claude mcp add-json agentcore-gateway '...'` using the Gateway URL from the Registry metadata. This writes the config, but **MCP servers are loaded at session startup** — the new server won't be available until you restart.

### Step 4: Restart and authenticate to Gateway

Exit and restart Claude Code to pick up the new Gateway server:

```bash
exit
claude
```

On startup, Claude Code now connects to **both** servers — you'll see **two Okta popups**:
1. Registry authentication (same as before)
2. **Gateway authentication** (new — second consent)

Run `/mcp` to verify both are connected:
- `open-insurance-registry` — connected
- `agentcore-gateway` — connected

### Step 5: Invoke tools via Gateway

```
> Look up insurance policy POL-10042
> Show me the claims summary
```

Now Claude Code invokes `PolicyLookup___lookup_policy` and `ClaimsData___query_claims` on the Gateway, with Cedar policies enforcing access control.

### The Full Picture

```
     ┌─────────────────────────────────────┐
     │           Claude Code                │
     └──────┬─────────────────────┬─────────┘
            │                     │
   Session 1│ (discover + add)   │ Session 2 (discover + invoke)
            │                     │
            ▼                     ▼
     ┌────────────┐       ┌─────────────┐
     │  Registry   │       │   Gateway    │
     │  (catalog)  │       │  (runtime)   │
     │             │       │              │
     │ search_     │       │ PolicyLookup │
     │ registry_   │  ───▶ │ ClaimsData   │
     │ records     │ found │              │
     └─────┬──────┘  here  └──────┬───────┘
           │                      │
     Okta popup #1          Okta popup #2
     (Registry auth)        (Gateway auth)
```

**Why this matters for regulated industries:**
- **Least-privilege access** — users only authenticate to systems they actually need
- **Audit trail** — each Okta popup = explicit consent to access that specific system
- **Different auth contexts** — different Gateways can require different Okta scopes, MFA levels, or authorization servers
- **Centralized discovery** — one Registry serves as the front door to all tools across the organization

### Re-adding the Gateway (after demo)

To restore the full setup from `03_claude_code_oauth_demo.ipynb`:

```bash
# The notebook printed the command in Cell 6 output — copy/paste it, or run:
claude mcp add-json agentcore-gateway '{"type":"http","url":"<gateway-url>","oauth":{...}}'
```

## Cleanup

Removes the Registry, its records, and the Claude Code MCP config. Does **not** remove the Gateway or Lambda targets (use `01_setup.ipynb` cleanup for that).

In [ ]:
# ⚠️  CLEANUP — This deletes the Registry and its records. Only run when done!

import json
import boto3
import subprocess

with open("gateway_config.json") as f:
    cfg = json.load(f)

region = cfg["aws_region"]
agentcore = boto3.client("bedrock-agentcore-control", region_name=region)

registry_id = cfg.get("registry_id", "")
registry_records = cfg.get("registry_records", {})

print("Cleaning up Registry resources...\n")

# 1. Delete registry records
for name, record_id in registry_records.items():
    try:
        agentcore.delete_registry_record(registryId=registry_id, recordId=record_id)
        print(f"  Deleted record: {name} ({record_id})")
    except Exception as e:
        print(f"  Skip record {name}: {e}")

# 2. Delete registry
if registry_id:
    try:
        agentcore.delete_registry(registryId=registry_id)
        print(f"  Deleted Registry: {registry_id}")
    except Exception as e:
        print(f"  Skip Registry: {e}")

# 3. Remove Registry from Claude Code
subprocess.run(["claude", "mcp", "remove", "open-insurance-registry"], capture_output=True)
print("  Removed open-insurance-registry from Claude Code")

# 4. Restore Gateway in Claude Code (removed in Cell 6 for Registry-first demo)
spa_client_id = cfg.get("okta_spa_client_id", "")
if spa_client_id:
    gateway_config_restore = {
        "type": "http",
        "url": cfg["gateway_url"],
        "oauth": {
            "clientId": spa_client_id,
            "callbackPort": cfg.get("claude_code_callback_port", 8400),
            "scope": "openid groups",
            "authorizationUrl": f"{cfg['okta_issuer']}/v1/authorize",
            "tokenUrl": f"{cfg['okta_issuer']}/v1/token",
        },
    }
    subprocess.run(["claude", "mcp", "remove", "agentcore-gateway"], capture_output=True)
    result = subprocess.run(
        ["claude", "mcp", "add-json", "agentcore-gateway", json.dumps(gateway_config_restore)],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print("  Restored agentcore-gateway in Claude Code")
    else:
        print(f"  Could not restore gateway: {result.stderr}")

# 5. Clean up gateway_config.json
for key in ["registry_id", "registry_arn", "registry_mcp_url", "registry_records"]:
    cfg.pop(key, None)

with open("gateway_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print("\n✅ Registry cleanup complete — Gateway restored in Claude Code")